# 02 — Prétraitement et découpage des données

> **Projet AutoMaze AI** — classification de 6 usagers de la route, entraînée de zéro.
> Règles du cours : pas de YOLO, pas de détecteur, **pas de poids préentraînés** (`pretrained=True` interdit).
> Les cellules `TODO` sont à écrire par l'équipe. Ne supprimez pas les questions : répondez-y en dessous, elles nourrissent le rapport.

**Objectif :** produire des recadrages propres et un découpage train / validation / test **figé**.

**Règles à respecter :**
- découpage **par image d'origine** : deux objets de la même photo vont dans le même jeu (sinon fuite de données) ;
- découpage **stratifié** : chaque classe garde à peu près les mêmes proportions dans les trois jeux ;
- moyenne / écart-type de normalisation calculés **sur le train uniquement** ;
- augmentation appliquée **au train uniquement** ;
- le test est mis de côté ici et **n'est plus regardé** avant le notebook 05.

## 0. Imports

In [ ]:
import json
import urllib.request

import pandas as pd
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import GroupShuffleSplit  # découpage par groupe (= par image d'origine)
from tqdm.auto import tqdm                             # barre de progression pour les téléchargements

In [ ]:
# --- Configuration commune (identique dans chaque notebook) ---
from pathlib import Path
import random

import numpy as np

# Racine du projet : ce notebook est dans notebooks/, donc on remonte d'un niveau.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"          # données locales, ignorées par Git
RAW_DIR = DATA_DIR / "raw"        # fichiers téléchargés tels quels (ex. annotations COCO)
CROPS_DIR = DATA_DIR / "crops"    # recadrages produits par le notebook 02
MODELS_DIR = ROOT / "models"      # poids + métadonnées (les .pt sont ignorés par Git)
FIGURES_DIR = ROOT / "reports" / "figures"  # figures à reprendre dans le rapport

for d in (RAW_DIR, CROPS_DIR, MODELS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Ordre des classes = source de vérité pour TOUT le projet (notebooks + simulateur).
# Ne le changez plus une fois l'entraînement commencé : l'indice 0 doit toujours désigner la même classe.
CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]
CLASSES_FR = ["Personne", "Vélo", "Voiture", "Moto", "Bus", "Camion"]

# Graine aléatoire : rend les résultats reproductibles (même découpage, mêmes initialisations).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1. Paramètres de prétraitement

Décidez-les **en équipe** et justifiez chacun dans le rapport (section 3).

In [ ]:
MIN_BOX_SIZE = None      # TODO(équipe) : côté minimal en pixels, issu du notebook 01
MAX_PER_CLASS = None     # TODO(équipe) : plafond d'exemples par classe (limite le déséquilibre et le temps)
CROP_MARGIN = None       # TODO(équipe) : marge autour de la bbox, en fraction (ex. 0.1 = 10 %)
INPUT_SIZE = None        # TODO(équipe) : taille d'entrée du modèle, ex. 64 (images INPUT_SIZE x INPUT_SIZE)
SPLITS = {"train": 0.70, "val": 0.15, "test": 0.15}  # à discuter

# Question : faut-il garder le ratio de l'objet (padding) ou l'étirer en carré ? Testez les deux visuellement.

## 2. Sélection des objets

Repartez du DataFrame du notebook 01 (rechargez le JSON ou sauvegardez le DataFrame en CSV dans le 01).

In [ ]:
# TODO(équipe) : filtrer iscrowd == 0 et la taille minimale, puis échantillonner au plus MAX_PER_CLASS par classe
# avec random_state=SEED.

## 3. Découpage train / validation / test par image

**Question :** pourquoi un découpage aléatoire objet par objet donnerait-il des scores trop optimistes ?

In [ ]:
# TODO(équipe) : utiliser GroupShuffleSplit (groups = image_id) deux fois : train vs reste, puis val vs test.
# Vérifier ensuite qu'aucun image_id n'apparaît dans deux jeux, et afficher le nombre par classe et par jeu.
# Sauvegarder le résultat dans DATA_DIR / "splits.csv" : ce fichier fige le découpage pour toute l'équipe.

## 4. Génération des recadrages

Organisation conseillée : `data/crops/<jeu>/<classe>/<id_annotation>.jpg`. Ce format est lisible directement par `torchvision.datasets.ImageFolder`.

In [ ]:
# TODO(équipe) : pour chaque ligne de splits.csv, télécharger l'image (mettre en cache dans RAW_DIR),
# découper la bbox avec la marge, borner les coordonnées à l'image, redimensionner à INPUT_SIZE, enregistrer.
# Conseil : testez sur 20 images avant de lancer le téléchargement complet.

## 5. Contrôle visuel

Affichez une grille de recadrages par classe et par jeu. Si vous voyez des erreurs, corrigez *avant* d'entraîner.

In [ ]:
# TODO(équipe) : grille d'exemples (par ex. 6 classes x 8 images) depuis data/crops/train.

## 6. Statistiques de normalisation (train uniquement)

In [ ]:
# TODO(équipe) : moyenne et écart-type par canal (R, G, B) sur les images de train, valeurs entre 0 et 1.
MEAN = None
STD = None

# Ces valeurs seront enregistrées dans models/metadata.json (notebook 04) : le simulateur les réutilisera.

## 7. Augmentation de données

Définissez ici les transformations, puis copiez-les **à l'identique** dans le notebook 04.

**Questions :** un retournement horizontal est-il réaliste pour une voiture ? Et un retournement vertical ?
Une rotation de 90° ? Que se passe-t-il si l'augmentation est trop forte ?

In [ ]:
from torchvision import transforms as T

# TODO(équipe) : compléter. Le train peut avoir de l'augmentation ; val et test JAMAIS.
train_transform = T.Compose([
    # ... augmentations choisies ...
    T.ToTensor(),
    # T.Normalize(MEAN, STD),
])
eval_transform = T.Compose([
    T.ToTensor(),
    # T.Normalize(MEAN, STD),
])

# TODO(équipe) : afficher une même image avec 8 augmentations différentes pour vérifier le rendu.

## 8. Conclusions

Résumez les paramètres retenus et le nombre final d'images par classe et par jeu (section 3 du rapport).